In [5]:
import pathlib
from fairscape_models.sql.models import ROCrateRegistration
from fairscape_lite.config import FilepathConfig, SQLConfig

from sqlalchemy import select, func
from sqlalchemy.orm import Session

In [6]:
# setup server config

storage = FilepathConfig("/tmp/server_content")
sql_db = SQLConfig(filepath = "/tmp/fairscape.db")
engine = sql_db.engine()


In [ ]:
# recieving a file
input_filepath = pathlib.Path("/mnt/data/Dataverse/U2OS/cm4ai_u2os_1_ImageDownloader.zip")
input_filename = input_filepath.name

In [4]:
outputFilepath = storage.outputPath(input_filename)

In [11]:
# get the rocrate metadata from the zip
from fairscape_models.utils import readCrate
import zipfile

In [83]:
def getZipInfo(zip_ref, path_within_zip):
	try:
		return zip_ref.getinfo(path_within_zip)
	except KeyError:
		return None

def findRootMetadata(input_filepath: pathlib.Path):
	with zipfile.ZipFile(str(input_filepath), 'r') as zip_ref:
		# is ro-crate-metadata.json at the top of the directory
		name = 'ro-crate-metadata.json'
		results = getZipInfo(zip_ref, name)	

		# within the zip a folder named as the stem with ro-crate-metadata.json
		if not results:
			name = f"{input_filepath.stem}/{name}"
			results = getZipInfo(zip_ref, name)	

		# default to search namelist
		if not results:
			namelist = zip_ref.namelist()
			matchingROCrates = [ elem for elem in namelist if 'ro-crate-metadata.json' in elem]

			if len(matchingROCrates) == 0:
				raise Exception()
			else:
				results = matchingROCrates[0]

		return results

In [23]:
%pip install ijson

Note: you may need to restart the kernel to use updated packages.


In [24]:
import ijson

In [52]:
def readOnlyMetadata(input_filepath):
	metadata_path_within_zip = findRootMetadata(input_filepath).filename
	with zipfile.ZipFile(str(input_filepath), 'r') as zip_ref:
		f = zip_ref.open(metadata_path_within_zip)
		objects = ijson.items(f, '@graph.item')
		rocrates = [ 
			{
				"@id": o.get("@id"), 
				"name": o.get("name"), 
				"version": o.get("version")
			} for o in objects if "https://w3id.org/EVI#ROCrate" in o.get('@type')]

	return rocrates[0]


In [55]:
metadata = readOnlyMetadata(input_filepath)

input_crate_guid = metadata.get("@id")

PosixPath('/tmp/content/cm4ai_u2os_1_ImageDownloader/v2/cm4ai_u2os_1_ImageDownloader.zip')

In [54]:
outputFilepath

PosixPath('/tmp/content/cm4ai_u2os_1_ImageDownloader.zip')

In [58]:
session = Session(engine)

In [77]:
# check to see if rocrate already exists
def determineVersion(session, crate_guid: str) -> int:
	max_version_query = select(func.max(ROCrateRegistration.version)).filter_by(guid=input_crate_guid)
	max_version_results = session.scalar(max_version_query)

	if not max_version_results:
		input_version = 1
	# if it exists set the version 
	else:
		input_version = max_version_results + 1

	return input_version


In [78]:
input_version = determineVersion(session, input_crate_guid)

In [79]:
input_file_stem = pathlib.Path(input_filename).stem
output_path = storage.storageFilepath / input_file_stem / f"v{input_version}" / input_filename

In [80]:
output_path

PosixPath('/tmp/content/cm4ai_u2os_1_ImageDownloader/v2/cm4ai_u2os_1_ImageDownloader.zip')

In [ ]:

new_registration = ROCrateRegistration(
	guid= input_crate_guid,
	filepath=str(output_path),
	version=input_version
)

session.add(new_registration)
session.flush()

In [ ]:
def writeOutputFile(input_file, output_path: pathlib.Path):
	""" Write out input ROCrate to Output Path"""
	input_file.seek(0)
	with output_path.open("w") as output_file:
		output_file.write(input_file)

## Register ROCrate

In [7]:
session = Session(engine)

In [10]:
list(session.scalars(select(ROCrateRegistration)))

[]

In [ ]:
# find the rocrate by guid optionally 
upload_id = 1
rocrate_query = select(ROCrateRegistration.filepath).filter_by(id=upload_id)
rocrate_results = session.scalars(rocrate_query).__next__()

In [19]:
crate_filepath = pathlib.Path(rocrate_results)

In [187]:
import zipfile
import ijson
import sqlalchemy

from fairscape_models.sql.models import (
	MetadataTypeEnumSQL, 
	IdentifiersSQL,
	ROCrateMetadataElemSQL,
	DatasetSQL,
	SoftwareSQL,
	ComputationSQL,
	ComputationUsedDatasetSQL,
	ComputationGeneratedDatasetSQL,
	MembershipSQL,
)
from fairscape_models.sql.utils import DetermineMetadataTypeSQL
from fairscape_lite.app import getZipInfo, findRootMetadata

In [32]:
if not crate_filepath.exists():
	raise Exception("Crate Filepath Not Found")


In [33]:
crate_filepath.name

'cm4ai_u2os_1_ImageDownloader.zip'

In [38]:
#zip_ref = zipfile.ZipFile(str(crate_filepath), 'r')
open_crate_file = crate_filepath.open('rb')

In [43]:
root_metadata_within_zip = findRootMetadata(input_filepath=open_crate_file, input_filename=crate_filepath.name).filename

In [195]:
root_metadata_within_zip

'cm4ai_u2os_1_ImageDownloader/ro-crate-metadata.json'

In [45]:
open_crate_file.close()

In [46]:
type(open_crate_file)

_io.BufferedReader

In [47]:
zip_ref = zipfile.ZipFile(str(crate_filepath), 'r')

In [ ]:
metadata_fp = zip_ref.open(root_metadata_within_zip)




In [ ]:
metadata_fp.seek(0)
graph_objects = ijson.items(metadata_fp, "@graph.item")

i = 0
for elem in graph_objects:

	if i > 10:
		break
	
	metadata_type = DetermineMetadataTypeSQL(elem.get("@type"))
	#print(elem.get("@type"))
	match metadata_type:
		case MetadataTypeEnumSQL.ROCRATE:
			# function for writing an rocrate
			pass
		case MetadataTypeEnumSQL.DATASET:
			# function for writing an rocrate
			pass
		case MetadataTypeEnumSQL.SOFTWARE:
			# function for writing an rocrate
			pass
		case MetadataTypeEnumSQL.COMPUTATION:
			# function for writing an rocrate
			pass
		case MetadataTypeEnumSQL.SCHEMA:
			# function for writing an rocrate
			pass
		

	
	
	i += 1



None
MetadataTypeEnumSQL.ROCRATE
MetadataTypeEnumSQL.SOFTWARE
MetadataTypeEnumSQL.DATASET
MetadataTypeEnumSQL.DATASET
MetadataTypeEnumSQL.COMPUTATION
MetadataTypeEnumSQL.DATASET
MetadataTypeEnumSQL.DATASET
MetadataTypeEnumSQL.DATASET
MetadataTypeEnumSQL.DATASET
MetadataTypeEnumSQL.DATASET


In [90]:
#len(list(dataset_generator))
test_dataset = next(dataset_generator)

In [97]:
test_dataset.get("author")

'Leah V. Schaffer, Mengzhou Hu, Gege Qian, Kyung-Mee Moon, Abantika Pal, Neelesh Soni, Andrew P. Latham, Laura Pontano Vaites, Dorothy Tsai, Nicole M. Mattson, Katherine Licon, Robin Bachelder, Anthony Cesnik, Ishan Gaur, Trang Le, William Leineweber, Aji Palar, Ernst Pulido, Yue Qin, Xiaoyu Zhao, Christopher Churas, Joanna Lenkiewicz, Jing Chen, Keiichiro Ono, Dexter Pratt, Peter Zage, Ignacia Echeverria, Andrej Sali, J. Wade Harper, Steven P. Gygi, Leonard J. Foster, Edward L. Huttlin, Emma Lundberg & Trey Ideker'

In [ ]:
import re

In [155]:
def transformDictAuthor(input_author)->list:
	if isinstance(input_author, str):
		author_list = [ auth_elem.lstrip(" ") for auth_elem in re.split(f'[,&]', input_author)]

	elif isinstance(input_author, list):
		# TODO processing and cleaning list of authors
		author_list = input_author

		# list of strings

		# list of dictionary
	return author_list




In [ ]:

selectDictKeys = lambda inputData, keyList: { key: inputData.get(key) for key in keyList}

ROCrateMetadataElemDictKeys = ["name", "description", "keywords", "version", "datePublished"]
DatasetDictKeys = ["name", "description", "keywords", "version", "datePublished"]
SoftwareDictKeys = ["name", "description", "keywords", "version", "datePublished"]
ComputationDictKeys = ["name", "description", "keywords", "dateCreated"]


# TODO preprocess all identifiers
def preprocessIdentifier(input_identifier):
	pass

def convertDatasets(dataset_generator):
	for ds in dataset_generator:
		yield {
			"guid": ds.get("@id"),
			"author": transformDictAuthor(ds.get("author")),
			"fileFormat": ds.get("format"),
			**selectDictKeys(ds, DatasetDictKeys)
		}

def writeDatasets(session, dataset_generator):
	session.execute(
		sqlalchemy.insert(DatasetSQL), 
		list(convertDatasets(dataset_generator))
	)	

def convertSoftware(gen):
	for sw in gen:
		yield {
			"guid": sw.get("@id"),
			"author": transformDictAuthor(sw.get("author")),
			"fileFormat": sw.get("format"),
			**selectDictKeys(sw, SoftwareDictKeys)
		}

def writeSoftware(session, gen):
	session.execute(
		sqlalchemy.insert(SoftwareSQL), 
		list(convertSoftware(gen))
	)	


def convertComputation(gen):
	for comp in gen:
		yield {
			"guid": comp.get("@id"),
			"author": transformDictAuthor(comp.get("runBy")),
			"fileFormat": comp.get("format"),
			**selectDictKeys(comp, ComputationDictKeys)
		}

def writeComputation(session, gen):
	session.execute(
		sqlalchemy.insert(ComputationSQL), 
		list(convertComputation(gen))
	)	


def convertROCrateMetadataElem(gen):
	for crate in gen:
		yield {
			"guid": crate.get("@id"),
			"author": transformDictAuthor(crate.get("author")),
			"fileFormat": crate.get("format"),
			**selectDictKeys(crate, ROCrateMetadataElemDictKeys)
		}

def writeROCrateMetadataElem(session, gen):
	session.execute(
		sqlalchemy.insert(ROCrateMetadataElemSQL), 
		list(convertROCrateMetadataElem(gen))
	)	



In [190]:
# clear database

for ds in session.scalars(sqlalchemy.select(DatasetSQL)):
	session.delete(ds)


for sw in session.scalars(sqlalchemy.select(SoftwareSQL)):
	session.delete(sw)

for comp in session.scalars(sqlalchemy.select(ComputationSQL)):
	session.delete(comp)

for crate in session.scalars(sqlalchemy.select(ROCrateMetadataElemSQL)):
	session.delete(crate)

for comp_used in session.scalars(sqlalchemy.select(ComputationUsedDatasetSQL)):
	session.delete(comp_used)

for comp_gen in session.scalars(sqlalchemy.select(ComputationGeneratedDatasetSQL)):
	session.delete(comp_gen)

for identifier in session.scalars(sqlalchemy.select(IdentifiersSQL)):
	session.delete(identifier)

for mem in session.scalars(sqlalchemy.select(MembershipSQL)):
	session.delete(mem)

session.commit()

In [160]:

metadata_fp.seek(0)
dataset_generator = (
	o for o in ijson.items(metadata_fp, "@graph.item")
	if DetermineMetadataTypeSQL(o.get("@type")) == MetadataTypeEnumSQL.DATASET
	)
len(list(dataset_generator))

#writeComputation(session, computation_generator)

20546

In [ ]:
# computation used dataset

In [ ]:
# computation generated dataset

In [151]:
session.commit()

In [158]:
import json

metadata_fp.seek(0)
metadata = json.load(metadata_fp)

In [159]:
[ ds for ds in metadata['@graph'] if DetermineMetadataTypeSQL(ds.get("@type")) == MetadataTypeEnumSQL.SOFTWARE]

[{'@id': 'https://fairscape.net/api/ark:59853/software-cellmaps-imagedownloader',
  'name': 'cellmaps_imagedownloader',
  '@type': ['prov:Entity', 'https://w3id.org/EVI#Software'],
  'author': 'Leah V. Schaffer, Mengzhou Hu, Gege Qian, Kyung-Mee Moon, Abantika Pal, Neelesh Soni, Andrew P. Latham, Laura Pontano Vaites, Dorothy Tsai, Nicole M. Mattson, Katherine Licon, Robin Bachelder, Anthony Cesnik, Ishan Gaur, Trang Le, William Leineweber, Aji Palar, Ernst Pulido, Yue Qin, Xiaoyu Zhao, Christopher Churas, Joanna Lenkiewicz, Jing Chen, Keiichiro Ono, Dexter Pratt, Peter Zage, Ignacia Echeverria, Andrej Sali, J. Wade Harper, Steven P. Gygi, Leonard J. Foster, Edward L. Huttlin, Emma Lundberg & Trey Ideker',
  'description': 'cellmaps_imagedownloader is a Python tool that downloads immunofluorescence microscopy images from the Human Protein Atlas (HPA). It retrieves IF images for specified genes and antibodies, organizing them by color channel (blue, red, green, yellow) and generating ge

In [ ]:

# write all elements

metadata_fp.seek(0)
# computation used dataset
computation_generator = (
	o for o in ijson.items(metadata_fp, "@graph.item") 
	if DetermineMetadataTypeSQL(o.get("@type")) == MetadataTypeEnumSQL.COMPUTATION
)

# computation generated dataset

# membership

# identifier table

session.commit()

In [165]:
def processIdentifierValue(input_list)->list[str]:
	""" Convert a list of Identifiers which may come as strings or dictionaries into a list of strings
	
	
	e.g.
		[
			"doi:9999/test",
			{"@id": "ark:59853/example"}	
		]	

		returns ["doi:9999/test", "@id": "ark:59853/example"]
	"""
	output_list = []
	for elem in input_list:
		if isinstance(elem, dict):
			output_list.append(elem.get("@id"))
		elif isinstance(elem, str): 
			output_list.append(elem)

	return output_list

In [188]:
def writeComputationProv(session, computation_generator):
	for comp in computation_generator:
		comp_guid = comp.get("@id")

		comp_generated = processIdentifierValue(comp.get("generated"))
		comp_used_dataset = processIdentifierValue(comp.get("usedDataset"))

		generated_prov_rows = [ {
			"computationGUID": comp_guid,
			"datasetGUID": gen_ds 
		} for gen_ds in comp_generated]

		used_prov_rows = [ {
			"computationGUID": comp_guid,
			"datasetGUID": gen_ds 
		} for gen_ds in comp_used_dataset]

		session.execute(
			sqlalchemy.insert(ComputationGeneratedDatasetSQL), 
			generated_prov_rows
		)	

		session.execute(
			sqlalchemy.insert(ComputationUsedDatasetSQL), 
			used_prov_rows
		)	

In [ ]:
# write all elements


# DATASET TABLE 
metadata_fp.seek(0)
dataset_generator = (
	{
			"guid": ds.get("@id"),
			"author": transformDictAuthor(ds.get("author")),
			"fileFormat": ds.get("format"),
			**selectDictKeys(ds, DatasetDictKeys)
		}
	for ds in ijson.items(metadata_fp, "@graph.item")
	if DetermineMetadataTypeSQL(ds.get("@type")) == MetadataTypeEnumSQL.DATASET
	)

session.execute(
	sqlalchemy.insert(DatasetSQL), 
	list(dataset_generator)
)	

# SOFTWARE TABLE
metadata_fp.seek(0)
software_generator = (
	{
			"guid": sw.get("@id"),
			"author": transformDictAuthor(sw.get("author")),
			"fileFormat": sw.get("format"),
			**selectDictKeys(sw, SoftwareDictKeys)
		}
	 for sw in ijson.items(metadata_fp, "@graph.item") 
	if DetermineMetadataTypeSQL(sw.get("@type")) == MetadataTypeEnumSQL.SOFTWARE
)

session.execute(
	sqlalchemy.insert(SoftwareSQL), 
	list(software_generator)
)	

# COMPUTATION TABLE
metadata_fp.seek(0)
computation_generator = (
	{
			"guid": comp.get("@id"),
			"author": transformDictAuthor(comp.get("runBy")),
			"fileFormat": comp.get("format"),
			**selectDictKeys(comp, ComputationDictKeys)
		}
	for comp in ijson.items(metadata_fp, "@graph.item") 
	if DetermineMetadataTypeSQL(comp.get("@type")) == MetadataTypeEnumSQL.COMPUTATION
)

session.execute(
	sqlalchemy.insert(ComputationSQL), 
	list(computation_generator)
)	

# PROV TABLES
metadata_fp.seek(0)
computation_generator = (
	o for o in ijson.items(metadata_fp, "@graph.item") 
	if DetermineMetadataTypeSQL(o.get("@type")) == MetadataTypeEnumSQL.COMPUTATION
)
writeComputationProv(session, computation_generator)

# IDENTIFIERS TABLE
metadata_fp.seek(0)
identifier_generator = (
	{
		"guid": o.get("guid"),
		"name": o.get("name"),
		"metadataType": DetermineMetadataTypeSQL(o.get("@type"))	
	} 
	for o in ijson.items(metadata_fp, "@graph.item") 
	)

session.execute(
	sqlalchemy.insert(IdentifiersSQL), 
	list(identifier_generator)
)	


# MEMBERSHIP TABLES
metadata_fp.seek(0)
rocrate_guid = [
	o.get("about", {}).get("@id") for o in ijson.items(metadata_fp, "@graph.item")  if o.get("@id") == "ro-crate-metadata.json"
	][0]

metadata_fp.seek(0)
member_guids = (
	{
		"parentGUID": rocrate_guid,
		"parentType": MetadataTypeEnumSQL.ROCRATE,
		"childGUID": o.get("@id"),
		"childType": DetermineMetadataTypeSQL(o.get("@type"))
	}
	for o in ijson.items(metadata_fp, "@graph.item")  if o.get("@id") != "ro-crate-metadata.json"
	)

session.execute(
	sqlalchemy.insert(MembershipSQL), 
	list(member_guids)
)	


session.commit()

In [ ]:
from collections import defaultdict
import sqlalchemy

TABLE_SPECS = {
				MetadataTypeEnumSQL.DATASET:     (DatasetSQL,     DatasetDictKeys,     "author"),
				MetadataTypeEnumSQL.SOFTWARE:    (SoftwareSQL,    SoftwareDictKeys,    "author"),
				MetadataTypeEnumSQL.COMPUTATION: (ComputationSQL, ComputationDictKeys, "runBy"),
}

def uploadMetadata(metadata_fp):
	""" Process the metadata upload in"""
	rows = defaultdict(list)
	computations, identifiers, members = [], [], []
	rocrate_guid = None

	metadata_fp.seek(0)
	for o in ijson.items(metadata_fp, "@graph.item"):
					guid = o.get("@id")
					if guid == "ro-crate-metadata.json":
									rocrate_guid = o.get("about", {}).get("@id")
									continue
					
					mtype = DetermineMetadataTypeSQL(o.get("@type"))   # classify once

					identifiers.append({"guid": guid, "name": o.get("name"), "metadataType": mtype})

					members.append({"childGUID": guid, "childType": mtype})

					spec = TABLE_SPECS.get(mtype)
					if spec:
									model, keys, author_field = spec
									rows[model].append({
													"guid": guid,
													"author": transformDictAuthor(o.get(author_field)),
													"fileFormat": o.get("format"),
													**selectDictKeys(o, keys),
									})
					if mtype == MetadataTypeEnumSQL.COMPUTATION:
									computations.append(o)

	if rocrate_guid is None:
					raise ValueError("ro-crate-metadata.json descriptor not found")
	for m in members:
					m["parentGUID"] = rocrate_guid
					m["parentType"] = MetadataTypeEnumSQL.ROCRATE

	# Writes
	for model in (DatasetSQL, SoftwareSQL, ComputationSQL):
					if rows[model]:
									session.execute(sqlalchemy.insert(model), rows[model])
	writeComputationProv(session, iter(computations))
	if identifiers:
					session.execute(sqlalchemy.insert(IdentifiersSQL), identifiers)
	if members:
					session.execute(sqlalchemy.insert(MembershipSQL), members)
	session.commit()

In [181]:
rocrate_guid

'https://fairscape.net/api/ark:59853/rocrate-cm4ai-image-downloader'

In [ ]:
#len(list(convertDatasets(dataset_generator)))

20546

In [ ]:

test_dataset

{'@id': 'https://fairscape.net/api/ark:59853/dataset-image-1521_a3_1',
 'name': 'MRPL19 IF Image (1521_A3_1_)',
 '@type': ['prov:Entity', 'https://w3id.org/EVI#Dataset'],
 'author': 'Leah V. Schaffer, Mengzhou Hu, Gege Qian, Kyung-Mee Moon, Abantika Pal, Neelesh Soni, Andrew P. Latham, Laura Pontano Vaites, Dorothy Tsai, Nicole M. Mattson, Katherine Licon, Robin Bachelder, Anthony Cesnik, Ishan Gaur, Trang Le, William Leineweber, Aji Palar, Ernst Pulido, Yue Qin, Xiaoyu Zhao, Christopher Churas, Joanna Lenkiewicz, Jing Chen, Keiichiro Ono, Dexter Pratt, Peter Zage, Ignacia Echeverria, Andrej Sali, J. Wade Harper, Steven P. Gygi, Leonard J. Foster, Edward L. Huttlin, Emma Lundberg & Trey Ideker',
 'description': 'Immunofluorescence image of MRPL19 (ensembl:ENSG00000115364) in U2OS cells (fold 1). Antibody: HPA046805. Subcellular locations: Mitochondria.',
 'version': '1.0',
 'contentUrl': 'http://images.proteinatlas.org/46805/1521_A3_1_blue_red_green.jpg',
 'isPartOf': [],
 'usedByCompu

StopIteration: 

In [ ]:
# functions for writing individual elements



In [ ]:
type(metadata_fp)

zipfile.ZipExtFile

In [58]:
type(session)

sqlalchemy.orm.session.Session

In [ ]:


async def process_metadata_file(
	metadata_zipped_file: zipfile.ZipExtFile,
	session: sqlalchemy.orm.session.Session
	):
	"""Async Processing to Write Files"""
	async for item in ijson.items(metadata_zipped_file, "@graph.item"):
		pass

[{'@id': 'https://fairscape.net/api/ark:59853/dataset-image-gene-node-attributes-with-locations',
  'name': 'Image Gene Node Attributes with Locations',
  '@type': ['prov:Entity', 'https://w3id.org/EVI#Dataset'],
  'author': 'Leah V. Schaffer, Mengzhou Hu, Gege Qian, Kyung-Mee Moon, Abantika Pal, Neelesh Soni, Andrew P. Latham, Laura Pontano Vaites, Dorothy Tsai, Nicole M. Mattson, Katherine Licon, Robin Bachelder, Anthony Cesnik, Ishan Gaur, Trang Le, William Leineweber, Aji Palar, Ernst Pulido, Yue Qin, Xiaoyu Zhao, Christopher Churas, Joanna Lenkiewicz, Jing Chen, Keiichiro Ono, Dexter Pratt, Peter Zage, Ignacia Echeverria, Andrej Sali, J. Wade Harper, Steven P. Gygi, Leonard J. Foster, Edward L. Huttlin, Emma Lundberg & Trey Ideker',
  'description': 'Gene node attribute file with subcellular location annotations. Contains 10,825 rows with gene name, Ensembl ID, antibody, image filename, image URL from proteinatlas.org, and subcellular locations. Used as input to cellmaps_imagedown

In [ ]:
def getKey(zip_ref, path_within_zip):
	try:
		return zip_ref.getinfo(path_within_zip)
	except KeyError:
		return None

In [ ]:
input_filepath.stem

'cm4ai_u2os_1_ImageDownloader'

In [ ]:
results

<ZipInfo filename='cm4ai_u2os_1_ImageDownloader/ro-crate-metadata.json' compress_type=deflate filemode='-rw-------' file_size=53821876 compress_size=1675536>

In [ ]:

crate_metadata = readCrate(input_filepath)

In [ ]:
# iterative json to find only the metadata elem


In [18]:
from fastapi import UploadFile

True